# Hover tooltips — `tooltip=` and `icon=` on the interactive views

Resting the pointer over a mark says what is under it: no click, no armed mode, and no change
to selection or to any linked view (PLANNING.md §7 **F1**). This is the check that defines
the feature as done: **hover a node and read it.**

Tooltips are **off by default**. Each view below starts with one on; on any view the
settings panel (`a`) has a `tooltip` row that cycles `off -> text -> icon` on `space`.
Needs a live kernel and the `interactive` extra (`pip install polars2svg[interactive]`).

In [ ]:
import pathlib, sys
_root = pathlib.Path('..').resolve()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import numpy as np
import polars as pl
import panel as pn
from polars2svg import Polars2SVG
import polars2svg; print(polars2svg.__file__)
pn.extension()
p2s = Polars2SVG()

# A small netflow-shaped frame: a few busy servers, a tail of clients, one hour of traffic.
rng   = np.random.default_rng(20261003)
hosts = [f'10.0.{i // 20}.{i % 20 + 1}' for i in range(60)]
w     = 1.0 / np.arange(1, len(hosts) + 1) ** 0.9
w    /= w.sum()
n     = 4000
t0    = np.datetime64('2026-10-03T09:00:00', 'us')
df = pl.DataFrame({
    'ts':    pl.Series(t0 + rng.integers(0, 3_600_000_000, n).astype('timedelta64[us]')),
    'sip':   rng.choice(hosts, n, p=w),
    'dip':   rng.choice(hosts, n, p=w[::-1] / w.sum()),
    'dport': rng.choice([80, 443, 53, 22, 445, 3389], n, p=[.3, .35, .15, .08, .07, .05]),
    'bytes': np.exp(rng.normal(6.0, 2.0, n)).round(),
}).filter(pl.col('sip') != pl.col('dip'))
df.height

## 1. Text tooltip on a `linkp`

Hover a node: it reports how many records it covers and the fields the graph is already
encoding (the two endpoints), summarising a field with many values as `N distinct`.

In [ ]:
lp = p2s.linkp(df=df, relationships=[('sip', 'dip')], wxh=(640, 480))
p2s.linkpi(lp, tooltip='text')

## 2. Icon tooltip: a 32x32 `xyp` of the hovered node's traffic

`icon=` takes a **rendered component** (not a live view, so `p2s.xypi(...)` is refused).
It is re-rendered against the records under the pointer, so the box is the icon's own
`wxh`. The first hover on a mark renders it; resting on the same mark again is cached.

In [ ]:
icon = p2s.xyp(df, 'ts', 'bytes', wxh=(32, 32))
p2s.linkpi(lp, tooltip='icon', icon=icon)

## 3. The same on a scatter

Hover a dot cluster; the text reports `x`, `y` and any `color=` field.

In [ ]:
p2s.xypi(p2s.xyp(df, 'ts', 'bytes', color='dport', wxh=(640, 360)), tooltip='text')